# 1 Normalization

## 1.1 LayerNorm

Given a hidden state $a$ from a Transformer, LayerNorm first computes its mean $\mu$ and standard variance $\sigma$, and then performs e-centering and re-scaling. The procedure is as follows:

$\bar{a}=\frac{a-\mu}\sigma*g_i$

where $g_i$ is a learnable scaling vector.



In [ ]:
import torch
import torch.nn as nn

class LayerNorm(nn.Module):

    def __init__(self,hidden_size,eps=1e-5):
        super().__init__()
        self.hidden_size=hidden_size
        self.eps=eps
        self.scaling=nn.Parameter(torch.ones(hidden_size))
        self.bias=nn.Parameter(torch.ones(hidden_size))
    
    def forward(self,x):
        mu=torch.mean(x,dim=-1,keepdim=True)
        sigma=torch.std(x,dim=-1,keepdim=True,unbiased=False)
        x=(x-mu)/(sigma+self.eps)
        return x*self.scaling
    
input=torch.randn(2,6,512)
my_LN=LayerNorm(512)
print(input)
print(my_LN(input))


## 1.2 RMSNorm
Different from LayerNorm, RMSNorm can be depicted as:

$\bar{a}=\frac{a}{rms}*g_i$

where $rms$ denotes root mean square.

In [ ]:
class RMSNorm(nn.Module):

    def __init__(self,hidden_size,eps=0):
        super().__init__()
        self.hidden_size=hidden_size
        self.eps=eps
        self.scaling=nn.Parameter(torch.ones(hidden_size))

    def forward(self,x):
        RMS=x.pow(2).mean(dim=-1, keepdim=True)
        x=x*torch.sqrt(self.eps+RMS)
        return x*self.scaling

input=torch.randn(2,6,512)
my_RMSNorm=RMSNorm(512)
print(input)
print(my_RMSNorm(input))


# 2 Positional encoding (PE)
Current research can be divided into two categories. Here, we only repoduce Sinusoidal PE and RoPE.
* Absolute PE
    1. **Learned PE** 
    2. **Sinusoidal PE** 
* Relative PE
    1. **RoPE**

## 2.1 Sinusoidal PE
It is originated from *Attention is all you need*, and can be depicted as:

$PE_{(pos,2i)}=\sin\left( \frac{pos} {10000^{2i/d_{model}}}\right)$

$PE_{(pos,2i+1)}=\cos\left( \frac{pos} {10000^{2i/d_{model}}} \right)$

Where $\quad i \in [0, d_{model}/2-1]$


In [ ]:
import torch
import torch.nn as nn
import numpy as np

# Sinusoidal PE
class My_SinusoidsPositionEmbedding(nn.Module):
    def __init__(self,max_length,channels):
        super().__init__()

        assert channels % 2 == 0
        freq=torch.Tensor([1/(10000**(2*i/channels)) for i in range(channels//2)])
        pos=torch.arange(max_length).float()
        x=pos[:,None]@freq[None,:]
        sin_tensor,cos_tensor=torch.sin(x),torch.cos(x)
        self.register_buffer(
            "positional_embedding",
            torch.cat([sin_tensor,cos_tensor],dim=-1),
            persistent=False)

    def forward(self,seg_len):

        return self.positional_embedding[:seg_len,:]

class Qwen_SinusoidsPositionEmbedding(nn.Module):
    def __init__(self, length, channels, max_timescale=10000):
        super().__init__()
        if channels % 2 != 0:
            raise ValueError("SinusoidsPositionEmbedding needs even channels input")
        log_timescale_increment = np.log(max_timescale) / (channels // 2 - 1)
        inv_timescales = torch.exp(-log_timescale_increment * torch.arange(channels // 2).float())
        scaled_time = torch.arange(length)[:, np.newaxis] * inv_timescales[np.newaxis, :]
        self.register_buffer(
            "positional_embedding",
            torch.cat([torch.sin(scaled_time), torch.cos(scaled_time)], dim=1),
            persistent=False,
        )

    def forward(self, seqlen: int):
        return self.positional_embedding[:seqlen, :]

if __name__=="__main__":
    # TODO in this example, some values are not identical? Figure out it!
    my_sinPE=My_SinusoidsPositionEmbedding(3,6)
    Qwen_sinPE=Qwen_SinusoidsPositionEmbedding(3,6)
    print("my",my_sinPE(3))
    print("Qwen",Qwen_sinPE(3))
    

## 2.2 RoPE
* TODO: fully understand RoPE

In [ ]:
# RoPE

# 3 Self-Attention

TODO using formulation to present self-attention

## 3.1 Group-Query Attention

In [ ]:
import torch
import torch.nn as nn

class QroupQueryAttention(nn.Module):
    
    # We leave the implementation of Qwen3ASRTextRMSNorm, pos_emb, mask, dropout, and kv Cache in the futrue

    def __init__(self,hidden_size,num_q_head,num_kv_head):
        super().__init__()
        
        self.hidden_size=hidden_size
        self.num_q_head=num_q_head
        self.num_kv_head=num_kv_head
        self.head_dim=hidden_size//num_q_head
        self.kv_group=num_q_head//num_kv_head
        self.scaling=self.head_dim**-0.5

        self.q=nn.Linear(hidden_size,self.head_dim*num_q_head)
        self.k=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.v=nn.Linear(hidden_size,self.head_dim*num_kv_head)
        self.o=nn.Linear(hidden_size,hidden_size)
		
    def forward(self,x):
        
    	# x [B,T,F]
        B,T,_=x.shape()
        
    	# forward QKV
        # TODO why T should be at the dim=2 rather than dim=1 ?
        Q=self.q(x).view(B,self.num_q_head,T,self.head_dim)
        K=self.k(x).view(B,self.num_kv_head,T,self.head_dim)
        V=self.v(x).view(B,self.num_kv_head,T,self.head_dim)
        
        # repeat K and V
        K=torch.repeat_interleave(K,dim=1,repeats=self.kv_group)
        V=torch.repeat_interleave(V,dim=1,repeats=self.kv_group)
        
        # comute attention scores
        attn_weights=Q@K.transpose(-2,-1)*self.scaling
        attn_weights=nn.functional.softmax(attn_weights,dim=-1)
        output=(attn_weights@V).view(B,T,-1)
        output=self.o(output)

        return output, attn_weights